In [1]:
import os
import pickle
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import numpy as np
import glob
import easyocr
import cv2
from imutils.video import FileVideoStream
from datetime import datetime, timedelta

In [3]:
load_dotenv()

True

In [ ]:
def time_converter(video_file, x_fac, y_fac):
    '''
    Gives the time of day (hour, minute, second) for each of the frames in a video

        Args:
            video_file: mp4 file of the salmon
            x_fac (int): specifies the scaling factor for the x-axis
            y_fac (int): specifies the scaling factor for the y-axis
        Returns:
            output (list): list of times in seconds, with index + 1 corresponding to the frame number
    '''
    fvs = FileVideoStream(video_file).start()
    output = []
    frame_count = 0
    while fvs.more():
        frame = fvs.read()
        if frame is None:
            break
        frame_count += 1
        frame = frame[20:76, 1443:1520]
        if frame_count == 1:
            # year, month, and day are arbitrary
            t1 = datetime(2025, 8, 1, int(video_file[82:84]), int(
                video_file[84:86]), int(video_file[86:88]))
            output.append(t1)
            continue
        frame = cv2.resize(frame, None, fx=x_fac, fy=y_fac)
        secs = str(t1.second).zfill(2) + \
            str((t1 + timedelta(seconds=1)).second).zfill(2)
        second = reader.readtext(frame, allowlist=secs, detail=0)
        if second:
            if second[0] == secs[2:]:
                if frame_count > 15:
                    if output[-1] == output[-14]:
                        t1 += timedelta(seconds=1)
                else:
                    t1 += timedelta(seconds=1)
        else:
            if frame_count >= 16:
                if output[-1] == output[-15]:
                    t1 += timedelta(seconds=1)
        output.append(t1)
    return output

In [ ]:
# "VIDEO FILES" in the .env document should link the folder with the video files
video_folder = os.environ.get("VIDEO FILES")

# Specify the desired date using format YYYYMMDD (without any / or -)
date = 
video_files = sorted(glob.glob(os.path.join(
    video_folder, f"RecM0A_DST{date}_*_*_0_*_*.mp4")))

outputs = {}

# Note that GPU is set to True for the reader by default
reader = easyocr.Reader(['en'])

# Choose an x and y scaling factor.
# I have found that setting both to 0.7 yields accurate results.
x_fac = 
y_fac = 

for i, video in enumerate(video_files):
    output = time_converter(video, x_fac, y_fac)
    outputs[i] = output

# Set the camera number that corresponds to the videos
camera_number = 
# Then you will export the file for the day as a pickle file
pickle.dump(outputs, open(f"true_times_{camera_number}_{date}.pkl", "wb"))